# Unit13 參數估計 | 練習題

本 Notebook 提供三類參數估計方法之化工應用練習題，每類各 5 題。

## 練習題清單

- **Part A**：`scipy.linalg.lstsq()` 線性最小平方法（題目 A1–A5）
- **Part B**：`scipy.optimize.curve_fit()` 非線性曲線擬合（題目 B1–B5）
- **Part C**：`scipy.optimize.least_squares()` 有界非線性估計（題目 C1–C5）

## 學習目標
- 根據模式特性（線性/非線性/有界）選擇適當的 Python 估計工具
- 建立設計矩陣（lstsq）、殘差函數（least_squares）及模式函數（curve_fit）
- 由 `pcov` 協方差矩陣推算 95% 置信區間並解讀其物理意義
- 視覺化擬合結果與殘差分布

---
### 0. 環境設定

In [ ]:
from pathlib import Path
import os

# ========================================
# 路徑設定 (兼容 Colab 與 Local)
# ========================================
UNIT_OUTPUT_DIR = 'Unit13_Practice'

try:
    from google.colab import drive
    IN_COLAB = True
    print("✓ 偵測到 Colab 環境，準備掛載 Google Drive...")
    drive.mount('/content/drive', force_remount=True)
except ImportError:
    IN_COLAB = False
    print("✓ 偵測到 Local 環境")

try:
    shortcut_path = '/content/ChemE-3502'
    os.remove(shortcut_path)
except (FileNotFoundError, OSError):
    pass

if IN_COLAB:
    source_path = Path('/content/drive/My Drive/Colab Notebooks/ChemE-3502')
    os.symlink(source_path, shortcut_path)
    shortcut_path = Path(shortcut_path)
    if source_path.exists():
        NOTEBOOK_DIR = shortcut_path / 'Unit13'
        OUTPUT_DIR   = NOTEBOOK_DIR / 'outputs' / UNIT_OUTPUT_DIR
        FIG_DIR      = OUTPUT_DIR / 'figs'
    else:
        print("⚠️ 找不到雲端 ChemE-3502 路徑，請確認資料夾名稱是否正確")
else:
    NOTEBOOK_DIR = Path.cwd()
    OUTPUT_DIR   = NOTEBOOK_DIR / 'outputs' / UNIT_OUTPUT_DIR
    FIG_DIR      = OUTPUT_DIR / 'figs'

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

print(f"\n✓ Notebook工作目錄: {NOTEBOOK_DIR}")
print(f"✓ 結果輸出目錄: {OUTPUT_DIR}")
print(f"✓ 圖檔輸出目錄: {FIG_DIR}")

---
### 1. 載入套件

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from scipy.linalg import lstsq
from scipy.optimize import curve_fit, least_squares

plt.rcParams.update({
    'figure.dpi': 100,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 12,
    'legend.fontsize': 10,
    'lines.linewidth': 2,
    'axes.unicode_minus': False,
})

print("✓ 套件載入完成")
import scipy, matplotlib
print(f"  numpy      版本: {np.__version__}")
print(f"  scipy      版本: {scipy.__version__}")
print(f"  matplotlib 版本: {matplotlib.__version__}")

---
## Part A：`scipy.linalg.lstsq()` 線性最小平方法練習（A1–A5）

| 題號 | 模式 | 重點 |
|------|------|------|
| A1 | $y = a + bx$ | 簡單線性回歸，計算 $J$ |
| A2 | $y = a + bx + cx^2$ | 二次多項式，設計矩陣建構 |
| A3 | $y = a\sin(x) + b\cos(x)$ | 三角函數基底，秩診斷 |
| A4 | $y = a \cdot x^b$（線性化） | 冪次律模式，對數線性化 |
| A5 | $y = a + be^x + ce^{-x}$ | 三參數指數模式，擬合曲線繪製 |

### A1：線性導熱模式 $Q = a + b \cdot T$

**題目**：某換熱器管壁在不同溫度下量測熱通量，實驗數據如下表。以線性模式 $Q = a + b \cdot T$ 估計截距 $a$ 與斜率 $b$，並計算目標函數 $J$。

| $T$ (K) | 300 | 350 | 400 | 450 | 500 | 550 | 600 |
|---------|-----|-----|-----|-----|-----|-----|-----|
| $Q$ (W/m²) | 124.5 | 161.8 | 199.3 | 238.1 | 274.7 | 311.9 | 349.2 |

**提示**：設計矩陣 $\mathbf{X} = [\mathbf{1},\ \mathbf{T}]$

In [ ]:
# ── A1：線性導熱模式 Q = a + b*T ──────────────────────────────
T_data = np.array([300, 350, 400, 450, 500, 550, 600], dtype=float)
Q_data = np.array([124.5, 161.8, 199.3, 238.1, 274.7, 311.9, 349.2])

# TODO: 建立設計矩陣 X = [1, T]
# 提示：np.column_stack([np.ones_like(T_data), T_data])


# TODO: 呼叫 lstsq(X, Q_data)，取出 theta, rank, sv
# 提示：theta, _, rank, sv = lstsq(X, Q_data)


# TODO: 解出截距 a 與斜率 b，計算目標函數 J = sum((Q_data - X @ theta)²)


# TODO: 印出 a, b, J 與設計矩陣秩，驗證是否滿秩 (rank == X.shape[1])


# TODO: 繪製散點圖與擬合曲線，儲存至 FIG_DIR / 'A1_linear_heat.png'


### A2：二次多項式管道壓降模式 $\Delta P = a + b \cdot v + c \cdot v^2$

**題目**：管道中不同流速 $v$ 下量測壓降 $\Delta P$，數據如下。以二次多項式模式估計三個參數，並驗證設計矩陣是否滿秩。

| $v$ (m/s) | 1.0 | 2.0 | 3.0 | 4.0 | 5.0 | 6.0 | 7.0 | 8.0 |
|----------|-----|-----|-----|-----|-----|-----|-----|-----|
| $\Delta P$ (kPa) | 2.6 | 7.0 | 13.5 | 23.1 | 34.8 | 48.3 | 65.1 | 83.9 |

**提示**：設計矩陣 $\mathbf{X} = [\mathbf{1},\ \mathbf{v},\ \mathbf{v}^2]$

In [ ]:
# ── A2：二次多項式壓降模式 ΔP = a + b*v + c*v² ─────────────────
v_data  = np.array([1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0])
dP_data = np.array([2.6, 7.0, 13.5, 23.1, 34.8, 48.3, 65.1, 83.9])

# TODO: 建立設計矩陣 X = [1, v, v²]
# 提示：np.column_stack([np.ones_like(v_data), v_data, v_data**2])


# TODO: 呼叫 lstsq(X, dP_data)，解出 a, b, c


# TODO: 計算目標函數 J，驗證設計矩陣秩 (rank == X.shape[1])


# TODO: 繪製 2 子圖：(1) 擬合曲線  (2) 殘差圖（stem plot）
# 儲存至 FIG_DIR / 'A2_quadratic_pressure.png'


### A3：三角函數基底模式 $y = a \sin(x) + b \cos(x)$

**題目**：以三角函數作為基底函數估計週期性信號參數，並驗證設計矩陣秩是否滿秩。

| $x$ (rad) | 0.5 | 1.0 | 1.5 | 2.0 | 2.5 | 3.0 |
|----------|-----|-----|-----|-----|-----|-----|
| $y$ | 2.53 | 3.76 | 4.06 | 3.37 | 1.91 | 0.13 |

**提示**：設計矩陣 $\mathbf{X} = [\sin(\mathbf{x}),\ \cos(\mathbf{x})]$ ；注意此模式無截距項

In [ ]:
# ── A3：三角函數基底  y = a*sin(x) + b*cos(x) ──────────────────
x_data = np.array([0.5, 1.0, 1.5, 2.0, 2.5, 3.0])
y_data = np.array([2.53, 3.76, 4.06, 3.37, 1.91, 0.13])

# TODO: 建立設計矩陣 X = [sin(x), cos(x)]（注意：無截距項）
# 提示：np.column_stack([np.sin(x_data), np.cos(x_data)])


# TODO: 呼叫 lstsq(X, y_data)，解出 a, b 與目標函數 J


# TODO: 印出設計矩陣秩與奇異值，驗證是否滿秩


# TODO: 計算等效振幅 A = sqrt(a²+b²) 與相位 φ = arctan2(b, a)


# TODO: 繪製擬合曲線，儲存至 FIG_DIR / 'A3_trig_model.png'


### A4：冪次律黏度模式線性化 $\mu = a \cdot T^b$

**題目**：液體黏度通常隨溫度降低，符合冪次律 $\mu = a \cdot T^b$（ $b < 0$ ）。對兩側取自然對數得 $\ln\mu = \ln a + b\ln T$，以 `lstsq()` 估計 $\ln a$ 與 $b$，再回推 $a = e^{\ln a}$，並計算原始空間中的目標函數 $J$。

| $T$ (K) | 300 | 320 | 340 | 360 | 380 | 400 | 420 |
|--------|-----|-----|-----|-----|-----|-----|-----|
| $\mu$ (cP) | 1.12 | 0.98 | 0.86 | 0.77 | 0.69 | 0.63 | 0.57 |

**提示**：線性化後 $y' = \ln\mu$，設計矩陣 $\mathbf{X}' = [\mathbf{1},\ \ln\mathbf{T}]$

In [ ]:
# ── A4：冪次律黏度模式線性化  μ = a * T^b ───────────────────────
T_data  = np.array([300, 320, 340, 360, 380, 400, 420], dtype=float)
mu_data = np.array([1.12, 0.98, 0.86, 0.77, 0.69, 0.63, 0.57])

# TODO: 線性化：令 y' = ln(μ)，X' = [1, ln(T)]
# 提示：y_lin = np.log(mu_data);  X_lin = np.column_stack([np.ones_like(T_data), np.log(T_data)])


# TODO: 呼叫 lstsq(X_lin, y_lin)，取出 ln_a 與 b，回推 a = exp(ln_a)


# TODO: 計算原始空間目標函數 J_orig 與對數空間目標函數 J_lin


# TODO: 繪製 2 子圖：(1) 原始空間 μ vs T  (2) 線性化 ln(μ) vs ln(T)
# 儲存至 FIG_DIR / 'A4_viscosity_powerlaw.png'


### A5：三參數指數模式 $y = a + b e^x + c e^{-x}$

**題目**：給定 8 組實驗數據，估計模式 $y = a + be^x + ce^{-x}$ 的三個參數，繪製擬合曲線與數據點比較圖，並報告目標函數 $J$。

| $x$ | 0.0 | 0.5 | 1.0 | 1.5 | 2.0 | 2.5 | 3.0 | 3.5 |
|-----|-----|-----|-----|-----|-----|-----|-----|-----|
| $y$ | 5.05 | 4.82 | 5.38 | 7.23 | 11.2 | 18.8 | 32.5 | 56.1 |

**提示**：設計矩陣 $\mathbf{X} = [\mathbf{1},\ e^\mathbf{x},\ e^{-\mathbf{x}}]$

In [ ]:
# ── A5：三參數指數模式  y = a + b*exp(x) + c*exp(-x) ────────────
x_data = np.array([0.0, 0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5])
y_data = np.array([5.05, 4.82, 5.38, 7.23, 11.2, 18.8, 32.5, 56.1])

# TODO: 建立設計矩陣 X = [1, exp(x), exp(-x)]
# 提示：np.column_stack([np.ones_like(x_data), np.exp(x_data), np.exp(-x_data)])


# TODO: 呼叫 lstsq(X, y_data)，解出 a, b, c 與目標函數 J


# TODO: 驗證設計矩陣秩，印出各點殘差


# TODO: 繪製 2 子圖：(1) 擬合曲線  (2) 殘差圖（stem plot）
# 儲存至 FIG_DIR / 'A5_three_param_exp.png'


---
## Part B：`scipy.optimize.curve_fit()` 非線性曲線擬合練習（B1–B5）

| 題號 | 模式 | 重點 |
|------|------|------|
| B1 | $y = a \cdot e^{-b x}$ | 指數衰減，`pcov` 與 95% CI |
| B2 | $y = a / (1 + b \cdot x)$ | 雙曲線型，比較不同初始猜測值 |
| B3 | $k = A \cdot e^{-E_a/(RT)}$ | Arrhenius 方程式，估計活化能 |
| B4 | $y = a \sin(bx + c) + d$ | 四參數正弦擬合，含 CI 標示 |
| B5 | $y = L / (1 + e^{-k(x-x_0)})$ | Logistic 成長，參數不確定性分析 |

### B1：指數衰減 $y = a \cdot e^{-b x}$ 與 95% 信賴區間

**題目**：8 組放射性衰減數據，以 `curve_fit()` 估計 $a$（初始活度）與 $b$（衰減常數），從協方差矩陣 `pcov` 計算各參數 95% CI，並在圖中標示誤差帶。

| $x$ (day) | 0 | 1 | 2 | 3 | 4 | 5 | 6 | 7 |
|-----------|---|---|---|---|---|---|---|---|
| $y$ (count/s) | 9.86 | 5.94 | 3.68 | 2.24 | 1.41 | 0.82 | 0.52 | 0.32 |

**提示**：`perr = np.sqrt(np.diag(pcov))`；95% CI 為 `popt ± 1.96 * perr`

In [ ]:
# ── B1：指數衰減  y = a * exp(-b*x) ─────────────────────────────
x_data = np.array([0, 1, 2, 3, 4, 5, 6, 7], dtype=float)
y_data = np.array([9.86, 5.94, 3.68, 2.24, 1.41, 0.82, 0.52, 0.32])

# TODO: 定義模型函數 exp_decay(x, a, b)


# TODO: 以 p0=[10.0, 0.5] 呼叫 curve_fit，取得 popt, pcov


# TODO: 計算 perr = sqrt(diag(pcov))，求 95% CI = 1.96 * perr
# 印出 a, b 與各自的 95% CI


# TODO: 計算目標函數 J 與半衰期 t½ = ln(2)/b


# TODO: 繪製擬合曲線與 95% CI 誤差帶，儲存至 FIG_DIR / 'B1_exp_decay.png'


### B2：雙曲線型衰減 $y = a / (1 + b \cdot x)$ 與初始猜測影響

**題目**：10 組濃度-時間數據符合雙曲線衰減。以三組不同初始猜測 `p0` 進行 `curve_fit()` 擬合，比較收斂結果，並討論初始猜測的影響。

| $x$ | 0 | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 | 9 |
|-----|---|---|---|---|---|---|---|---|---|---|
| $y$ | 5.12 | 3.38 | 2.58 | 2.08 | 1.76 | 1.52 | 1.36 | 1.22 | 1.12 | 1.04 |

**提示**：嘗試 `p0 = [5, 0.5]`、`[10, 2]`、`[1, 0.1]` 各一次，觀察是否收斂至相同解

In [ ]:
# ── B2：雙曲線型衰減  y = a / (1 + b*x) ─────────────────────────
x_data = np.array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9], dtype=float)
y_data = np.array([5.12, 3.38, 2.58, 2.08, 1.76, 1.52, 1.36, 1.22, 1.12, 1.04])

# TODO: 定義模型函數 hyperbolic_decay(x, a, b)


# TODO: 以三組不同 p0 分別呼叫 curve_fit，記錄各自的 a, b, J
# p0_list = [[5.0, 0.5], [10.0, 2.0], [1.0, 0.1]]
# 提示：用 for 迴圈，try/except 處理不收斂情況


# TODO: 比較三組結果，討論是否收斂至相同解


# TODO: 繪圖：將三組擬合曲線疊在同一張圖上
# 儲存至 FIG_DIR / 'B2_hyperbolic_p0_compare.png'


### B3：Arrhenius 方程式 $k = A \cdot e^{-E_a/(RT)}$ 與活化能估計

**題目**：6 組反應速率常數—溫度數據符合 Arrhenius 方程式，其中氣體常數 $R = 8.314\ \mathrm{J\ mol^{-1}\ K^{-1}}$。以 `curve_fit()` 直接擬合非線性形式，計算 $A$ 與 $E_a$ 的 95% CI。

| $T$ (K) | 300 | 320 | 340 | 360 | 380 | 400 |
|---------|-----|-----|-----|-----|-----|-----|
| $k$ (s⁻¹) | 0.00145 | 0.00631 | 0.02312 | 0.07392 | 0.2103 | 0.5419 |

**提示**：初始猜測可先對數線性化得到 $A_0$、 $E_{a,0}$

In [ ]:
# ── B3：Arrhenius  k = A * exp(-Ea/(R*T)) ───────────────────────
R = 8.314  # J/(mol·K)
T_data = np.array([300, 320, 340, 360, 380, 400], dtype=float)
k_data = np.array([0.00145, 0.00631, 0.02312, 0.07392, 0.2103, 0.5419])

# TODO: 定義 arrhenius(T, A, Ea) 函數


# TODO: 對數線性化求初始猜測 A0, Ea0
# 提示：ln(k) = ln(A) - Ea/(R*T) → X' = [1, 1/T]
# A0 = exp(theta[0]),  Ea0 = -theta[1] * R


# TODO: 呼叫 curve_fit(arrhenius, T_data, k_data, p0=[A0, Ea0])
# 取得 popt, pcov，計算 A_fit, Ea_fit 及 95% CI


# TODO: 繪製 2 子圖：(1) k vs T 擬合  (2) Arrhenius plot ln(k) vs 1/T
# 儲存至 FIG_DIR / 'B3_arrhenius.png'


### B4：四參數正弦 $y = a \sin(bx + c) + d$ 與信賴區間帶

**題目**：12 組週期性信號數據，以 `curve_fit()` 估計振幅 $a$、角頻率 $b$、相位 $c$、偏移量 $d$，並在圖中繪製 $\pm 1\sigma$ 誤差帶。

| $x$ | 0 | 0.5 | 1.0 | 1.5 | 2.0 | 2.5 | 3.0 | 3.5 | 4.0 | 4.5 | 5.0 | 5.5 |
|-----|---|-----|-----|-----|-----|-----|-----|-----|-----|-----|-----|-----|
| $y$ | 1.92 | 3.48 | 4.63 | 4.81 | 4.02 | 2.55 | 0.91 | -0.38 | -0.81 | -0.23 | 1.23 | 2.91 |

**提示**：可從數據估計週期後設定 `b0 ≈ 2π/T`；`a0` 取極差的一半；`d0` 取均值

In [ ]:
# ── B4：四參數正弦  y = a*sin(b*x + c) + d ──────────────────────
x_data = np.array([0, 0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0, 5.5])
y_data = np.array([1.92, 3.48, 4.63, 4.81, 4.02, 2.55, 0.91,
                   -0.38, -0.81, -0.23, 1.23, 2.91])

# TODO: 定義 sine4(x, a, b, c, d) 函數


# TODO: 估計初始猜測
# a0 = (max - min) / 2,  d0 = mean,  b0 = 2π/T（T ≈ 5）,  c0 = 0.5


# TODO: 呼叫 curve_fit，取得 popt, pcov，計算 perr = sqrt(diag(pcov))
# 印出 a, b, c, d 與各自的 ±1σ 誤差，計算週期 T = 2π/b


# TODO: 繪製含 ±1σ 誤差帶的擬合圖
# 儲存至 FIG_DIR / 'B4_sine4param.png'


### B5：Logistic 成長曲線 $y = L / (1 + e^{-k(x-x_0)})$ 與參數不確定性

**題目**：10 組細胞增殖數據呈 S 型成長，模式為 $y = L/(1+e^{-k(x-x_0)})$，其中 $L$ 為最大容量，$k$ 為成長率，$x_0$ 為半飽和點。以 `curve_fit()` 擬合後，用 `pcov` 分析各參數不確定性，並計算成長率最大點。

| $x$ (day) | 0 | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 | 9 |
|-----------|---|---|---|---|---|---|---|---|---|---|
| $y$ (×10⁶ cells) | 0.12 | 0.29 | 0.71 | 1.73 | 3.84 | 6.42 | 8.31 | 9.28 | 9.76 | 9.93 |

**提示**：最大成長率 $\mathrm{d}y/\mathrm{d}x|_{\max} = k \cdot L / 4$，出現在 $x = x_0$ 時

In [ ]:
# ── B5：Logistic 成長  y = L / (1 + exp(-k*(x-x0))) ────────────
x_data = np.array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9], dtype=float)
y_data = np.array([0.12, 0.29, 0.71, 1.73, 3.84, 6.42, 8.31, 9.28, 9.76, 9.93])

# TODO: 定義 logistic(x, L, k, x0) 函數


# TODO: 估計初始猜測
# L0 = max * 1.05,  k0 = 1.0,  x00 = x 最大增量點（np.argmax(np.diff(y_data))）


# TODO: 呼叫 curve_fit，取得 popt, pcov，計算 95% CI = 1.96 * sqrt(diag(pcov))


# TODO: 計算最大成長率 = k*L/4，出現在 x = x0


# TODO: 計算參數相關矩陣：corr = pcov / np.outer(sqrt(diag(pcov)), sqrt(diag(pcov)))


# TODO: 繪製 2 子圖：(1) Logistic 成長曲線  (2) 參數不確定性棒形圖（perr）
# 儲存至 FIG_DIR / 'B5_logistic_growth.png'


---
## Part C：`scipy.optimize.least_squares()` 有界限制非線性擬合（C1–C5）

| 題號 | 模式 | 重點 |
|------|------|------|
| C1 | $y = a(1 - e^{-bx})$ | 有界 vs 無界比較 |
| C2 | $Q = Q_{\max}KC/(1+KC)$ | Langmuir 吸附，參數有物理意義下界 |
| C3 | $y = ax^2/(b+x^2)$ | 飽和型，殘差圖分析 |
| C4 | $C = C_0 e^{-kt}$ | 有界 vs curve_fit 的 pcov 比較 |
| C5 | $y = ae^{-bx}\sin(cx+d)$ | 四參數阻尼振盪，`result.cost` 解析 |

### C1：飽和增長 $y = a(1-e^{-bx})$ — 有界 vs 無界

**題目**：8 組吸附飽和數據，模式 $y = a(1-e^{-bx})$（ $a > 0$，$b > 0$）。分別以無界（`bounds=(-inf,inf)`）與有界（`bounds=([0,0],[10,5])`）進行 `least_squares()` 求解，比較兩者結果，討論邊界設定的影響。

| $x$ | 0 | 0.5 | 1.0 | 1.5 | 2.0 | 3.0 | 4.0 | 5.0 |
|-----|---|-----|-----|-----|-----|-----|-----|-----|
| $y$ | 0.00 | 1.37 | 2.48 | 3.34 | 3.98 | 4.95 | 5.47 | 5.75 |

In [ ]:
# ── C1：飽和增長  y = a*(1 - exp(-b*x)) ─────────────────────────
x_data = np.array([0, 0.5, 1.0, 1.5, 2.0, 3.0, 4.0, 5.0])
y_data = np.array([0.00, 1.37, 2.48, 3.34, 3.98, 4.95, 5.47, 5.75])

# TODO: 定義 sat_growth(x, a, b) 與殘差函數 residuals(params, x, y)
# 提示：residuals(params, x, y) 回傳 sat_growth(x, *params) - y


# TODO: 無界求解：least_squares(residuals, x0=[6, 0.5], args=(x_data, y_data))
# J = 2 * result.cost（因 cost = 0.5 * sum(r²)）


# TODO: 有界求解：least_squares(..., bounds=([0, 0], [10, 5]), ...)


# TODO: 比較兩者的 a, b, J 與 result.message


# TODO: 繪圖：兩條擬合曲線疊在同一張圖上
# 儲存至 FIG_DIR / 'C1_sat_growth_bounds.png'


### C2：Langmuir 吸附模式 $Q = Q_{\max}KC/(1+KC)$

**題目**：8 組平衡吸附數據符合 Langmuir 模式，其中 $Q_{\max}$ 為最大吸附量（mg/g）， $K$ 為吸附平衡常數（L/mg）。以 `least_squares()` 設定物理意義邊界 $Q_{\max} \in [50, 300]$、 $K \in [0, 10]$ 進行有界擬合。

| $C$ (mg/L) | 1 | 2 | 5 | 10 | 20 | 40 | 80 | 120 |
|-----------|---|---|---|----|----|----|----|-----|
| $Q$ (mg/g) | 12.3 | 21.8 | 44.1 | 68.5 | 101.3 | 135.8 | 162.4 | 178.2 |

In [ ]:
# ── C2：Langmuir 吸附  Q = Qmax*K*C / (1 + K*C) ────────────────
C_data = np.array([1, 2, 5, 10, 20, 40, 80, 120], dtype=float)
Q_data = np.array([12.3, 21.8, 44.1, 68.5, 101.3, 135.8, 162.4, 178.2])

# TODO: 定義 langmuir(C, Qmax, K) 與殘差函數 res_langmuir(params, C, Q)


# TODO: Langmuir 線性化求初始猜測
# 令 y' = C/Q，X' = [1, C] → lstsq 求解
# Qmax0 = 1 / theta[1],  K0 = theta[1] / theta[0]


# TODO: least_squares(res_langmuir, x0, bounds=([50, 0], [300, 10]), ...)
# 輸出 Qmax_fit, K_fit, J = 2*result.cost, result.message
# 計算半飽和濃度 C½ = 1/K（Q = Qmax/2 時）


# TODO: 繪製 2 子圖：(1) Q vs C 等溫線  (2) 線性化圖 C/Q vs C
# 儲存至 FIG_DIR / 'C2_langmuir.png'


### C3：飽和動力學 $y = ax^2/(b+x^2)$ 與殘差圖

**題目**：9 組酵素反應數據符合高次飽和動力學 $y = ax^2/(b+x^2)$（ $a > 0$，$b > 0$）。以 `least_squares()` 有界擬合，繪製擬合曲線與殘差分析圖。

| $x$ | 0 | 1 | 2 | 3 | 5 | 8 | 12 | 18 | 25 |
|-----|---|---|---|---|---|---|----|----|----|
| $y$ | 0.0 | 2.1 | 6.8 | 12.3 | 22.4 | 31.6 | 37.8 | 41.2 | 43.5 |

In [ ]:
# ── C3：飽和動力學  y = a*x^2 / (b + x^2) ───────────────────────
x_data = np.array([0, 1, 2, 3, 5, 8, 12, 18, 25], dtype=float)
y_data = np.array([0.0, 2.1, 6.8, 12.3, 22.4, 31.6, 37.8, 41.2, 43.5])

# TODO: 定義 hill2(x, a, b) 與殘差函數 res_hill2(params, x, y)


# TODO: least_squares(res_hill2, [50, 10], bounds=([0, 0], [np.inf, np.inf]), ...)
# 取出 a_fit, b_fit，計算 J = 2*result.cost
# 計算半飽和 x½ = sqrt(b)


# TODO: 分析 result.fun（殘差向量）：均值、標準差、max|r|


# TODO: 繪製 2 子圖：(1) 擬合曲線（含 Ymax 水平線）  (2) 殘差圖（scatter + vlines）
# 儲存至 FIG_DIR / 'C3_sat_kinetics.png'


### C4：一階衰減 $C = C_0 e^{-kt}$ — 有界 `least_squares` 與 `curve_fit` 比較

**題目**：7 組濃度衰減數據，以有界 `least_squares()`（ $C_0 \in [0.5,2.0]$，$k \in [0,2]$ ）求解，再以 `curve_fit()` 擬合取得 `pcov`，比較兩者估計值，並分析 `pcov` 的物理意義。

| $t$ (min) | 0 | 1 | 2 | 3 | 4 | 6 | 8 |
|-----------|---|---|---|---|---|---|---|
| $C$ (mol/L) | 1.05 | 0.78 | 0.58 | 0.43 | 0.32 | 0.18 | 0.10 |

In [ ]:
# ── C4：一階衰減  C = C0 * exp(-k*t) ────────────────────────────
t_data = np.array([0, 1, 2, 3, 4, 6, 8], dtype=float)
C_data = np.array([1.05, 0.78, 0.58, 0.43, 0.32, 0.18, 0.10])

# TODO: 定義 first_order(t, C0, k) 與殘差函數 res_fo(params, t, C)


# TODO: least_squares(res_fo, [1.0, 0.3], bounds=([0.5, 0], [2.0, 2.0]), ...)
# 取得 C0_ls, k_ls，計算 J_ls = 2*result.cost


# TODO: curve_fit(first_order, t_data, C_data, p0=x0) 取得 pcov_cf
# 分析 pcov 對角線：Var(C0) = pcov[0,0], Var(k) = pcov[1,1], cov(C0,k) = pcov[0,1]


# TODO: 計算兩種方法的半衰期 t½ = ln(2)/k 並比較


# TODO: 繪製半對數擬合比較圖（兩條曲線）
# 提示：ax.set_yscale('log')
# 儲存至 FIG_DIR / 'C4_first_order_decay.png'


### C5：阻尼振盪 $y = ae^{-bx}\sin(cx+d)$ 與 `result.cost` 解析

**題目**：12 組阻尼振盪數據，以 `least_squares()` 設定邊界 $a \in [0,5]$、$b \in [0,3]$、$c \in [0,10]$、$d \in [-\pi,\pi]$ 進行四參數擬合。將 `least_squares` 最佳解作為 `curve_fit` 的初始猜測，取得 `pcov` 並比較 `result.cost` 與 $J$ 的關係。

| $x$ | 0 | 0.5 | 1.0 | 1.5 | 2.0 | 2.5 | 3.0 | 3.5 | 4.0 | 4.5 | 5.0 | 5.5 |
|-----|---|-----|-----|-----|-----|-----|-----|-----|-----|-----|-----|-----|
| $y$ | 0.00 | 1.82 | 2.30 | 1.61 | 0.45 | -0.59 | -1.04 | -0.87 | -0.36 | 0.19 | 0.48 | 0.45 |

**重點**：`result.cost = 0.5 * sum(r²) = J/2`

In [ ]:
# ── C5：阻尼振盪  y = a*exp(-b*x)*sin(c*x + d) ──────────────────
x_data = np.array([0, 0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0, 5.5])
y_data = np.array([0.00, 1.82, 2.30, 1.61, 0.45,
                  -0.59, -1.04, -0.87, -0.36, 0.19, 0.48, 0.45])

# TODO: 定義 damped_osc(x, a, b, c, d) 與殘差函數 res_damp(params, x, y)


# TODO: least_squares(res_damp, [2.5, 0.4, 2.0, 0.2],
#       bounds=([0, 0, 0, -np.pi], [5, 3, 10, np.pi]), method='trf')
# 取得 a_ls, b_ls, c_ls, d_ls，計算 J_ls = 2*result.cost
# 解釋關係：result.cost = 0.5 * sum(r²) = J/2


# TODO: 以 least_squares 最佳解作為 p0，呼叫 curve_fit 取得 pcov 與 perr
# 提示：用 try/except 包住，以防 curve_fit 不收斂


# TODO: 繪製 2 子圖：
#   (1) 含包絡線（±a*exp(-b*x)）的擬合圖
#   (2) result 物件解析：印出 cost, J, nfev, status, jac.shape, ||r||
# 儲存至 FIG_DIR / 'C5_damped_osc.png'
